# Projet — Exploitation du dateset CL-Drive

## Estimation de la charge cognitive du conducteur

Ce TP vient après les TD1, TD2, TD3 et TD4.

Les TD ont déjà permis de travailler :

- la compréhension du papier CL-Drive ;
- le protocole expérimental ;
- la segmentation en fenêtres de 10 s ;
- le prétraitement EEG ;
- l'extraction des features EEG.

Le point de départ du TP est donc le dossier généré à la fin du TD4 :

```text
EEG_Features_10s/
```

Ce TP ne revient pas sur le calcul des features. Il exploite les features déjà extraites pour construire un pipeline d'apprentissage automatique.

## Objectif du TP

Construire un pipeline complet :

```text
EEG_Features_10s
→ Normalized_Features_10s/EEG
→ Normalized_Features_10s_With_Label/EEG
→ Dataset EEG supervisé
→ Classification de la charge cognitive
→ Évaluation
→ Interprétation
```

Dans un premier temps, on se limite à l'EEG uniquement.

La multimodalité, c'est-à-dire l'ajout de ECG, EDA et Gaze, sera proposée uniquement comme extension à la fin du sujet.

## 1. Structure attendue des dossiers

Avant de commencer, le dossier de travail doit contenir au minimum :

```text
Data/
├── EEG/ID_x
│   ├── ... fichiers level_1, level_2, ..., level_9
│   ├── ... fichiers baseline
│   └── ... fichiers filtered_*
│
├── EEG_Features_10s/
│   ├── ID1_EEG_features.csv
│   ├── ID2_EEG_features.csv
│   └── ...
│
├── Labels/
│   ├── ID1.csv
│   ├── ID2.csv
│   └── ...
```

Le TP va générer deux nouveaux dossiers :

```text
Data/
├── Normalized_Features_10s/
│   └── EEG/
│       ├── norm_ID1_EEG_features.csv
│       ├── norm_ID2_EEG_features.csv
│       └── ...
│
├── Normalized_Features_10s_With_Label/(avec colonnes Level et Label)
│   └── EEG/
│       ├── norm_ID1_EEG_features.csv
│       ├── norm_ID2_EEG_features.csv
│       └── ...
```

## Question 

Pourquoi ne faut-il pas entraîner directement les modèles sur les fichiers `EEG_Features_10s`  ?

### Réponse 

...

In [4]:
from pathlib import Path

# TODO : adapter ce chemin à votre organisation locale.
BASE_PATH = Path("Data")

EEG_FEATURE_DIR = BASE_PATH / "EEG_Features_10s"
LABEL_DIR = BASE_PATH / "Labels"
NORMALIZED_ROOT = BASE_PATH / "Normalized_Features_10s"
NORMALIZED_EEG_DIR = NORMALIZED_ROOT / "EEG"
LABELED_ROOT = BASE_PATH / "Normalized_Features_10s_With_Label"
LABELED_EEG_DIR = LABELED_ROOT / "EEG"

NORMALIZED_EEG_DIR.mkdir(parents=True, exist_ok=True)
LABELED_EEG_DIR.mkdir(parents=True, exist_ok=True)

METADATA_COLUMNS = ["Participant", "File", "Window", "Start_Time", "End_Time", "Channel"]

## 2. Normalisation des features EEG

À la fin du TD4, chaque fichier CSV contient des features EEG calculées sur des fenêtres de 10 secondes.

La normalisation doit suivre deux étapes :

### Étape 1 — Normalisation par la baseline du sujet

Pour chaque sujet, les fichiers de baseline servent à calculer une valeur moyenne de référence pour chaque feature :

$$
\mu_{baseline}^{(s,f)} = \frac{1}{N}\sum_{i=1}^{N} x_i^{(s,f)}
$$

où :

- $s$ désigne le sujet ;
- $f$ désigne la feature ;
- $x_i^{(s,f)}$ désigne la valeur de la feature pendant la baseline.

Chaque valeur de feature dans les fichiers de tâche est ensuite divisée par la moyenne de baseline correspondante :

$$
x_{norm}^{(s,f)} = \frac{ x^{(s,f)} }{ \mu_{baseline}^{(s,f)} }
$$

### Étape 2 — Standardisation z-score

On applique ensuite une standardisation :

$$
z = \frac{x - \mu}{\sigma}
$$

Cela permet d’obtenir des features centrées et réduites. Cette étape devra toutefois être réalisée plus loin dans le pipeline, après la séparation des données entre les ensembles d’entraînement et de test (voir section 8 ci-dessous).

## Question

Quel est l'intérêt de la normalisation par baseline dans des signaux physiologiques ?

### Réponse 

...

In [5]:
def get_feature_columns(df, metadata_columns=METADATA_COLUMNS):
    """
    Retourne les colonnes numériques correspondant aux features.
    Les colonnes de métadonnées ne doivent pas être normalisées.
    """
    # TODO : sélectionner les colonnes numériques et retirer les métadonnées.


def compute_baseline_averages(feature_dir):
    """
    Calcule, pour chaque Participant, la moyenne de baseline de chaque feature.

    Indications :
    - parcourir les fichiers CSV de feature_dir ;
    - garder uniquement les fichiers dont le nom contient 'baseline' ;
    - lire chaque fichier avec pandas.read_csv ;
    - identifier le Participant avec df['Participant'].iloc[0] ;
    - calculer la moyenne de chaque feature ;
    """
    baseline_avgs = {}
    # TODO : compléter.
    return baseline_avgs


def normalize_by_baseline(df, participant_id, baseline_avgs):
    """
    Divise chaque feature par sa moyenne de baseline pour le sujet considéré.
    """
    # TODO : compléter.
    return df_norm



def run_eeg_normalization():
    """
    Génère les fichiers du dossier :
    Normalized_Features_10s/EEG
    à partir du dossier :
    EEG_Features_10s
    """
    # TODO : calculer les profils baseline.
    # TODO : parcourir les fichiers non-baseline.
    # TODO : sauvegarder sous le nom norm_<nom_fichier>.csv dans NORMALIZED_EEG_DIR.
    

In [ ]:
run_eeg_normalization()

## 3. Vérification du dossier `Normalized_Features_10s/EEG`

Après exécution de la normalisation, vérifiez que le dossier contient bien des fichiers `norm_*.csv`.

## Question

Pourquoi les fichiers de baseline ne sont-ils pas copiés dans le dossier normalisé final ?

### Réponse 

...

In [ ]:
# Vérification du dossier Normalized_Features_10s/EEG


## 4. Ajout des colonnes `Level` et `Label`

Les fichiers normalisés ne contiennent pas encore la cible d'apprentissage.

Il faut maintenant associer chaque fenêtre de 10 secondes à son score PAAS.

Les labels sont stockés dans le dossier :

```text
Labels/
```

Chaque fichier de labels correspond à un sujet, par exemple :

```text
Labels/ID1.csv
Labels/ID2.csv
...
```

Dans ces fichiers, on suppose une structure du type :

| time | lvl_1 | lvl_2 | ... | lvl_9 |
|---:|---:|---:|---|---:|
| 10 | 2 | 3 | ... | 5 |
| 20 | 2 | 4 | ... | 6 |
| ... | ... | ... | ... | ... |

Pour une fenêtre d'indice `Window`, le temps associé est :

$$
time = (Window + 1) \times 10
$$

Le niveau du scénario est extrait du nom du fichier avec une expression régulière :

```text
level_1 → Level = 1
level_2 → Level = 2
...
level_9 → Level = 9
```

Le score PAAS est ensuite récupéré dans la colonne :

```text
lvl_<Level>
```

Exemple : si `Level = 4`, on lit la colonne `lvl_4`.


In [ ]:
def extract_level_from_filename(file_name):
    """
    Extrait le niveau de scénario à partir du nom de fichier.

    Exemple :
    filtered_level_3.csv → 3
    norm_filtered_level_8.csv → 8
    """
    # TODO : utiliser re.search.


def get_label_for_row(row, labels_df):
    """
    Retourne le score PAAS correspondant à une ligne de features.

    Indications :
    - récupérer Window ;
    - calculer time_stamp = (Window + 1) * 10 ;
    - récupérer Level ;
    - construire label_col = f'lvl_{Level}' ;
    - chercher dans labels_df la ligne où labels_df['time'] == time_stamp ;
    - retourner la valeur de label_col.
    """
    # TODO : compléter.


def attach_labels_eeg():
    """
    Génère les fichiers du dossier :
    Normalized_Features_10s_With_Label/EEG

    Chaque fichier de sortie doit contenir deux nouvelles colonnes :
    - Level
    - Label
    """
    # TODO : parcourir les fichiers normalisés.
    # TODO : identifier le Participant.
    # TODO : charger Labels/<Participant>.csv.
    # TODO : ajouter Level.
    # TODO : ajouter Label.
    # TODO : supprimer les lignes sans Label.
    # TODO : sauvegarder dans LABELED_EEG_DIR.

In [ ]:
attach_labels_eeg()

## 5. Vérification du dossier `Normalized_Features_10s_With_Label/EEG`

Le dossier final doit contenir des fichiers CSV avec au moins :

- les métadonnées : `Participant`, `File`, `Window`, `Channel`, `Start_Time` et `End_Time` ;
- les features EEG normalisées ;
- la colonne `Level` ;
- la colonne `Label`.

## Question

Quelle est la différence entre `Level` et `Label` dans ce TP ? Pourquoi faut-il ajouter à la fois `Level` et `Label` ?

### Réponse

Level désigne le numéro du scénario de conduite (1 à 9), c'est une information sur la complexité de la tâche.

Label est le score PAAS déclaré par le participant, c'est la vérité terrain pour la classification.

Il faut conserver les deux car Level permet de retrouver l'origine d'une fenêtre et d'analyser les résultats par scénario, tandis que Label est la cible utilisée pour entraîner le modèle.

In [6]:
import pandas as pd

# Vérification du dossier Normalized_Features_10s_With_Label/EEG
files = sorted(LABELED_EEG_DIR.glob("*.csv"))
print(f"Nombre de fichiers : {len(files)}")

required_cols = ["Participant", "File", "Window", "Channel", "Start_Time", "End_Time", "Level", "Label"]

for f in files:
    df_check = pd.read_csv(f)
    missing = [col for col in required_cols if col not in df_check.columns]
    if missing:
        print(f"{f.name} : colonnes manquantes -> {missing}")
    else:
        print(f"{f.name} : OK ({len(df_check)} lignes, {len(df_check.columns)} colonnes)")

Nombre de fichiers : 0


## 6. Construction du dataset EEG supervisé

Une fois les fichiers normalisés et labellisés générés, on peut les concaténer pour construire un tableau unique.

Chaque ligne représente une fenêtre EEG de 10 secondes pour un canal.

On construit ensuite deux problèmes possibles :

### Classification binaire

| Score PAAS | Classe |
|---:|---|
| 1 à 4 | faible |
| 5 à 9 | élevée |

### Classification ternaire, extension

| Score PAAS | Classe |
|---:|---|
| 1 à 3 | faible |
| 4 à 6 | moyenne |
| 7 à 9 | élevée |

Dans ce TP, l'objectif principal est la classification binaire.

In [9]:
import pandas as pd

def load_labeled_eeg_dataset():
    """
    Concatène tous les fichiers CSV du dossier Normalized_Features_10s_With_Label/EEG.
    """
    # TODO : parcourir LABELED_EEG_DIR, lire les CSV, concaténer avec pd.concat.
    dfs = []
    for f in sorted(LABELED_EEG_DIR.glob("*.csv")):
        dfs.append(pd.read_csv(f))
    return pd.concat(dfs, ignore_index=True)


df = load_labeled_eeg_dataset()
print(df.shape)
df.head()

ValueError: No objects to concatenate

In [10]:
# Création des cibles de classification.
df["Label_Binary"] = df["Label"].apply(lambda x: 0 if x <= 4 else 1)

# Extension ternaire éventuelle.
df["Label_Ternary"] = df["Label"].apply(lambda x: 0 if x <= 3 else (1 if x <= 6 else 2))

print("Distribution binaire :")
print(df["Label_Binary"].value_counts())
print("\nDistribution ternaire :")
print(df["Label_Ternary"].value_counts())

NameError: name 'df' is not defined

## 7. Préparation de la matrice d'apprentissage

On doit séparer :

- les métadonnées ;
- les features numériques EEG ;
- la cible d'apprentissage.

## Question

Pourquoi ne faut-il pas inclure `Participant`, `File`, `Window`, `Level` ou `Label` dans les features du modèle ?

### Réponse 

Il ne faut pas inclure Participants, File, Window, Level et Label dans les features du modèle car ce ne sont pas des mesures physiologiques. Elles servent à identifier l'origine ou la cible d'une fenêtre. Cela fausserait le modèle de les inclure car, par exemple, si le modèle apprend que le participant X a souvent une charge cognitive élevée, il mémorisera au lieu de généraliser. En particulier, le Label ne doit pas être inclus car c'est la variable qu'on cherche à prédire.

In [ ]:
# Préparation des données d’entraînement

METADATA_COLUMNS = ["Participant", "File", "Window", "Channel", "Start_Time", "End_Time", "Level", "Label", "Label_Binary", "Label_Ternary"]

feature_cols = [col for col in df.columns if col not in METADATA_COLUMNS]

X = df[feature_cols].values
y = df["Label_Binary"].values

print("Nombre d'exemples :", X.shape[0])
print("Nombre de features :", X.shape[1])
print("Exemples de features :", feature_cols[:5])

## 8. Classification EEG — premiers modèles

On teste plusieurs modèles classiques :

- LDA ;
- SVM ;
- Random Forest ;
- KNN ;
- Naive Bayes ;
- Decision Tree ;
- AdaBoost ;
- MLP.

La normalisation `StandardScaler` est placée dans le `sklearn.pipeline.Pipeline` pour éviter une fuite de données entre apprentissage et test. Il faut ajuster le `StandardScaler` uniquement sur les données d’entraînement :

`scaler.fit_transform(X_train)`

Puis appliquer la transformation aux données de test avec :

`scaler.transform(X_test)`

In [1]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.neural_network import MLPClassifier

models = {
  "LDA": Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LinearDiscriminantAnalysis()),
  ]),
  "SVM": Pipeline([
    ("scaler", StandardScaler()),
    ("clf", SVC())
  ]),
  "Random Forest": Pipeline([
    ("scaler", StandardScaler()),
    ("clf", RandomForestClassifier())
  ]),
  "KNN": Pipeline([
    ("scaler", StandardScaler()),
    ("clf", KNeighborsClassifier())
  ]),
  "Naive Bayes": Pipeline([
    ("scaler", StandardScaler()),
    ("clf", GaussianNB())
  ]),
  "Decision Tree": Pipeline([
    ("scaler", StandardScaler()),
    ("clf", DecisionTreeClassifier())
  ]),
  "AdaBoost": Pipeline([
    ("scaler", StandardScaler()),
    ("clf", AdaBoostClassifier())
  ]),
  "MLP": Pipeline([
    ("scaler", StandardScaler()),
    ("clf", MLPClassifier(max_iter=500))
  ]),
}

## 9. Évaluation par validation croisée et par sujet

Deux évaluations sont demandées :

### 10-fold cross-validation

Les segments sont répartis en 10 folds stratifiés. Cette évaluation est utile pour comparer les modèles, mais elle peut mélanger les sujets entre apprentissage et test.

### Leave-One-Subject-Out, LOSO

Un sujet est laissé de côté pour le test, tandis que le modèle est entraîné sur les autres sujets. Cette stratégie d’évaluation est plus réaliste, car elle permet de tester la capacité de généralisation du modèle sur un conducteur jamais vu auparavant. L’opération est ensuite répétée sur l’ensemble des sujets disponibles afin d’obtenir une évaluation plus robuste.

## Question

Pourquoi le LOSO est-il souvent plus difficile que le 10-fold classique ?

### Réponse 

En 10-fold, les données d'un même sujet peuvent se retrouver à la fois dans le train et dans le test, ce qui facilite la tâche du modèle. En LOSO, le modèle est évalué sur un sujet qu'il n'a jamais vu, ce qui est bien plus difficile car les signaux EEG varient fortement d'une personne à l'autre.

## 10. Interprétation et discussion

Répondez aux questions suivantes dans le notebook :

1. Quel modèle obtient le meilleur F1-score en 10-fold ?
2. Quel modèle obtient le meilleur F1-score en LOSO ?
3. Les performances chutent-elles en LOSO ? Pourquoi ?
4. Les classes sont-elles équilibrées ?
5. Les résultats obtenus avec EEG seul vous semblent-ils suffisants pour une application réelle ?
6. Quelles limites voyez-vous à l'utilisation des labels subjectifs PAAS ?
7. Quelles améliorations proposeriez-vous ?

## 11. Mini-système d'adaptation

À partir de la prédiction du modèle, on peut simuler une décision d'adaptation.

Exemple :

| Prédiction | Décision |
|---|---|
| charge faible | interface normale |
| charge élevée | simplification de l'interface |
| charge élevée persistante | alerte conducteur |

## Question 

Pourquoi faut-il être prudent avant de déclencher une alerte sur une seule prédiction ?

### Réponse 

...

In [ ]:
def decision_system(...):
    """
    Transforme les prédictions en décision d'adaptation.
    
    """




## 12. Extension optionnelle — vers la multimodalité

Le cœur du TP est volontairement limité à l'EEG.

Une extension possible consiste à reproduire les mêmes étapes pour les autres modalités :

```text
ECG_Features_10s → Normalized_Features_10s/ECG → Normalized_Features_10s_With_Label/ECG
EDA_Features_10s → Normalized_Features_10s/EDA → Normalized_Features_10s_With_Label/EDA
Gaze_Features_10s → Normalized_Features_10s/Gaze → Normalized_Features_10s_With_Label/Gaze
```

Puis à fusionner les features :

```text
EEG + ECG
EEG + EDA
EEG + Gaze
EEG + ECG + EDA + Gaze
```

La fusion la plus simple est une concaténation des colonnes de features pour des fenêtres correspondant au même sujet, au même niveau et au même indice de fenêtre.

## Question

Pourquoi la multimodalité peut-elle améliorer la détection de la charge cognitive ?

### Réponse 

...